In [1]:
import pandas as pd
import json

In [2]:
file = "../data/external/first_100_selected_examples_without_docstrings_base_model_og_prompt_V2_instruct_comparison_with_position_info_post_process.json"

In [3]:
def create_input_prompt_prefix(task_prompt, test_list, mode = "base"):
    prompt = (
            "You are an expert Python programmer, and here is your task: "
            f"{task_prompt} Your code should pass these tests:\n\n"
            + "\n".join(test_list) + "\nWrite your code below starting with \"```python\" and ending with \"```\".\n```python\n"
        )
    if mode == "instruct":
        prompt = (
            "You are an expert Python programmer, and here is your task: "
            f"{task_prompt} Your code should pass these tests:\n\n"
            + "\n".join(test_list) + "\nWrite your code, without docstrings, below starting with \"```python\" and ending with \"```\".\n```python\n"
        )
    
    return prompt

In [4]:
import ast

class Normalizer(ast.NodeTransformer):
    def __init__(self):
        self.var_map = {}
        self.func_map = {}
        self.class_map = {}
        self.counter = 0

    def _rename(self, name, mapping):
        if name not in mapping:
            mapping[name] = f"id_{len(mapping)}"
        return mapping[name]

    def visit_Name(self, node):
        node.id = self._rename(node.id, self.var_map)
        return node

    def visit_arg(self, node):
        node.arg = self._rename(node.arg, self.var_map)
        return node

    def visit_FunctionDef(self, node):
        node.name = self._rename(node.name, self.func_map)
        self.generic_visit(node)
        return node

    def visit_ClassDef(self, node):
        node.name = self._rename(node.name, self.class_map)
        self.generic_visit(node)
        return node

def normalize(code):
    try:
        tree = ast.parse(code)
        normalized = Normalizer().visit(tree)
        return ast.dump(normalized, annotate_fields=True, include_attributes=False)
    except:
        return None

In [5]:
def check_if_abstract_tree_same(base_response, instruct_response):
    return normalize(base_response) == normalize(instruct_response)

In [6]:
import difflib

def similarity(code1, code2):
    return difflib.SequenceMatcher(None, code1, code2).ratio()

In [10]:
sample_base_response = "def sum_series(n):\n    return n - 2 * (n // 2)"
sample_instruct_response = "def sum_series(n):\n  total = 0\n  for i in range(n // 2):\n    total += n - 2 * i\n  return total\n"

In [23]:
def _extract_code_part(input_prefix_text, suffix_text):
    full_text = input_prefix_text + suffix_text
    return "def" + full_text.split("```python\n")[-1].split("def")[1]

def _ascertain_two_codes_similar(expected_code, steered_code, threshold = 0.95, verbose = False):
    normalized_expected_code = normalize(expected_code)
    normalized_steered_code = normalize(steered_code)
    if verbose:
        print("Check for full similarity", normalized_expected_code == normalized_steered_code)
        print("Check for whether crosses threshold", similarity(normalized_expected_code, normalized_steered_code))
    return normalized_expected_code == normalized_steered_code or similarity(normalized_expected_code, normalized_steered_code) > threshold

In [16]:
_ascertain_two_codes_similar(sample_base_response, sample_instruct_response)

False

In [25]:
sample_base_response = "def sum_series(n):\n    return n - 2 * func(n - 2)"
alt_base_response = """def sum_series(m):\n    return m ** 0.5 - 2 * func (m // 2)"""
_ascertain_two_codes_similar(sample_base_response, alt_base_response, verbose = True)

Check for full similarity False
Check for whether crosses threshold 0.9375


False

In [27]:
sample_base_response = """
def sum_series(n):
    sum = 0
    for i in range(5):
        sum += i
    return sum
"""
alt_base_response = """
def sum_ser(b):
    s = 0
    for i in range(0, 5):
        s = s + i
    return s
"""
_ascertain_two_codes_similar(sample_base_response, alt_base_response, verbose = True)

Check for full similarity False
Check for whether crosses threshold 0.6421232876712328


False

### Use the Zhang-ShaSha distance of actual similarity based on some threshold.
-- https://zhang-shasha.readthedocs.io/en/latest/

In [31]:
from zss import simple_distance, Node

def to_zss(node):
    if not isinstance(node, ast.AST):
        return None
    children = []
    for f in node._fields:
        value = getattr(node, f)
        if isinstance(value, list):
            for child in value:
                if isinstance(child, ast.AST):
                    children.append(to_zss(child))
        elif isinstance(value, ast.AST):
            children.append(to_zss(value))
    return Node(node.__class__.__name__, children=children)

In [38]:
def normalize_ast(code):
    try:
        tree = ast.parse(code)
        norm = Normalizer().visit(tree)
        ast.fix_missing_locations(norm)
        return norm
    except Exception as e:
        return None

In [39]:
def _check_zss_sim(code1, code2, threshold = 4, verbose = False):
    node1 = normalize_ast(code1)
    node2 = normalize_ast(code2)
    distance = simple_distance(to_zss(node1), to_zss(node2))
    if verbose:
        print("Check for distance", distance)
    return distance <= 4

In [41]:
_check_zss_sim(sample_base_response, alt_base_response, verbose = True)

Check for distance 5.0


False

In [43]:
sample_base_response = "def sum_series(n):\n    return n - 2 * (n // 2)"
sample_instruct_response = "def sum_series(n):\n  total = 0\n  for i in range(n // 2):\n    total += n - 2 * i\n  return total\n"
_check_zss_sim(sample_base_response, sample_instruct_response, verbose = True)

Check for distance 25.0


False

In [45]:
sample_base_response = """
def sum_series(n):
    sum = 0
    for i in range(5):
        sum += i
    return sum
"""
alt_base_response = """
def sum_ser(b):
    s = 0
    for i in range(0, 5):
        s = s + i
    return s
"""
_check_zss_sim(sample_base_response, alt_base_response, verbose = True)

Check for distance 5.0


False

In [46]:
sample_base_response = """
def sum_series(n):
    sum = 0
    for i in range(5):
        sum += i
    return sum
"""
alt_base_response = """
def sum_ser(b):
    s = 0
    for i in range(5):
        s += i
    return s
"""
_check_zss_sim(sample_base_response, alt_base_response, verbose = True)

Check for distance 0.0


True

In [47]:
sample_base_response = """
def format_string(list_of_string, intended_format):
    t_string = []
    for i in range(len(list_of_string)):
        t_string.append(list_of_string[i].format(intended_format))
    return t_string
"""
alt_base_response = """
def format_string(b_list, b_format):
    return [u.format(b_format) for u in b_list]
"""
_check_zss_sim(sample_base_response, alt_base_response, verbose = True)

Check for distance 29.0


False

In [48]:
sample_base_response = """
def format_string(list_of_string, intended_format):
    return [list_of_string[i].format(intended_format) for i in range(len(list_of_string))]
"""
alt_base_response = """
def format_string(b_list, b_format):
    return [u.format(b_format) for u in b_list]
"""
_check_zss_sim(sample_base_response, alt_base_response, verbose = True)

Check for distance 10.0


False

In [49]:
sample_base_response = """
def tetrahedral_number(n):
    return (n * (n + 1) * (n + 2)) // 6
"""
alt_base_response = """
def tetrahedral_number(n):
    return 1 + 2 * (n - 1) 
"""
_check_zss_sim(sample_base_response, alt_base_response, verbose = True)

Check for distance 12.0


False

In [52]:
sample_base_response = """
def tetrahedral_number(n):
    return (n * (n + 1) * (n + 2)) // 6
"""
alt_base_response = """
def tetrahedral_number(n):
    return (2 * n * (n + 1) * (n + 2)) // 3
"""

_check_zss_sim(sample_base_response, alt_base_response, verbose = True)
_ascertain_two_codes_similar(sample_base_response, alt_base_response, verbose = True)

Check for distance 3.0
Check for full similarity False
Check for whether crosses threshold 0.952561669829222


True

In [54]:
sample_base_response = """
def tetrahedral_number(n):
    return (n * (n + 1) * (n + 2)) // 6
"""
alt_base_response = """
def tetrahedral_number(n):
    return n * (2 * n + 1) * (n + 2) / 6
"""

_check_zss_sim(sample_base_response, alt_base_response, verbose = True)
_ascertain_two_codes_similar(sample_base_response, alt_base_response, verbose = True)

Check for distance 4.0
Check for full similarity False
Check for whether crosses threshold 0.9494756911344138


False

In [ ]:
sample_base_response = """
def tetrahedral_number(n):
    return (n * (n + 1) * (n + 2)) // 6
"""
alt_base_response = """
def tetrahedral_number(n):
    return n * (2 * n + 1) * (n + 2) / 6
"""

_check_zss_sim(sample_base_response, alt_base_response, verbose = True)
_ascertain_two_codes_similar(sample_base_response, alt_base_response, verbose = True)

In [ ]:


def check_if_answers_are_different_after_signature(entry, threshold = 0.95):
    instruct_output = entry["instruct_code"]
    base_output = entry["model_output"]
    task = entry["prompt"]
    test_list = entry["test_list"]
    base_prefix = create_input_prompt_prefix(task, test_list, mode = "base")
    instruct_prefix = create_input_prompt_prefix(task, test_list, mode = "instruct")
    base_code = _extract_code_part(base_prefix, base_output)
    instruct_code = _extract_code_part(instruct_prefix, instruct_output)
    normalized_base_code = normalize(base_code)
    normalized_instruct_code = normalize(instruct_code)
    if normalized_base_code != None and normalized_instruct_code != None:
        return normalized_base_code != normalized_instruct_code
    else:
        return similarity(base_code, instruct_code) < threshold